# Day 26 - Shuffle Broadcast Join and Skew

**Topic:** Shuffle Broadcast Join and Skew  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจว่า shuffle เกิดตอนไหน, broadcast join ช่วยอะไร, และ data skew ทำให้ Spark job ช้าได้อย่างไร

วันนี้เราจะเรียน performance mindset ที่เจอบ่อยใน PySpark pipeline จริง โดยเฉพาะตอน join และ aggregation ว่าทำไมบาง job ถึงช้า ทั้งที่ code ดูเรียบง่าย เช่น `join`, `groupBy`, `orderBy` หรือ `repartition` เพราะเบื้องหลังอาจมี shuffle, join strategy ที่ไม่เหมาะสม หรือข้อมูลกระจุกอยู่ที่ key เดียวจนเกิด data skew


## Goal of the day

1. แยกให้ออกว่า operation ไหนมีโอกาสเป็น **wide transformation** และทำให้เกิด **shuffle**
2. อธิบายได้ว่า **shuffle** คือการ move data ข้าม partition / executor เพื่อจัดข้อมูลใหม่ตาม key
3. อ่าน Spark execution plan จาก `.explain()` เพื่อเปรียบเทียบ normal join กับ **broadcast join** ได้
4. ตรวจสัญญาณของ **data skew** จาก key distribution และ partition distribution ได้
5. ใช้ basic Spark performance techniques เช่น broadcast join สำหรับ small dimension join และ salting เพื่อลดปัญหา data skew ใน aggregation ได้


## Why it matters in production

ใน production pipeline งานที่ช้ามาก ๆ มักไม่ได้ช้าเพราะ `select` หรือ `withColumn` ธรรมดา แต่ช้าเพราะ Spark ต้องย้ายข้อมูลจำนวนมากระหว่าง executors หรือมีบาง partition หนักกว่า partition อื่นมาก

เรื่องนี้สำคัญเพราะ:

- `join`, `groupBy`, `distinct`, `orderBy`, `repartition` มักทำให้เกิด shuffle
- shuffle มักใช้ network, disk spill, memory และ executor time สูงกว่างานแบบ narrow transformation
    - Disk spill คือกรณีที่ Spark ใช้ memory ไม่พอระหว่าง shuffle / sort / aggregation เลยต้องเขียนข้อมูลชั่วคราวลง disk แทน ทำให้งานช้ากว่าการประมวลผลใน memory
- join กับ dimension table เล็ก ๆ อาจเร็วขึ้นมากถ้าใช้ broadcast join อย่างเหมาะสม
- data skew ทำให้บาง task ใช้เวลานานกว่า task อื่นมาก แม้จำนวน records รวมไม่ได้ใหญ่มาก
- การอ่าน `.explain(mode="formatted")` ช่วยให้เห็น join strategy เช่น `SortMergeJoin`, `BroadcastHashJoin`, `Exchange`

Production takeaway:

> Slow Spark jobs often come from shuffle, bad join strategy, or data skew. อย่าดูแค่จำนวน rows รวม ให้ดู data distribution และ physical plan ด้วย


## Key concepts

| Concept | Meaning |
|---|---|
| Narrow transformation | Transformation ที่แต่ละ output partition ใช้ข้อมูลจาก input partition เดิมเป็นหลัก เช่น `select`, `filter`, `withColumn` |
| Wide transformation | Transformation ที่ต้องจัดข้อมูลใหม่ข้าม partition เช่น `groupBy`, `join`, `distinct`, `orderBy`, `repartition` |
| Shuffle | การ move data ข้าม partitions / executors เพื่อให้ records ที่มี key เดียวกันไปอยู่ด้วยกัน |
| Exchange | คำที่มักเห็นใน physical plan เมื่อ Spark ต้อง shuffle หรือ broadcast data |
| Join strategy | วิธีที่ Spark เลือกใช้เพื่อ join เช่น sort-merge join, broadcast hash join |
| Broadcast join | การส่ง table เล็กไปยังทุก executor เพื่อให้ join กับ table ใหญ่ได้โดยไม่ต้อง shuffle ข้อมูลฝั่ง table ใหญ่ |
| Broadcast hint | การบอก Spark ว่า DataFrame ฝั่งหนึ่งควรถูก broadcast เพื่อใช้ broadcast join เช่น `F.broadcast(df_dim)` หรือ `df_dim.hint("broadcast")` |
| Data skew | ข้อมูลกระจุกที่ key หรือ partition บางอันมากผิดปกติ ทำให้บาง task หนักกว่า task อื่น |
| Skewed key | Key ที่มี records เยอะกว่า key อื่นมาก เช่น customer คนเดียวมี transaction ส่วนใหญ่ของ dataset และอาจทำให้เกิด data skew หลัง `groupBy` หรือ `join` |
| Salting | การเพิ่ม salt key ชั่วคราวเพื่อกระจาย records ของ skewed key ออกเป็นหลายกลุ่ม ลดโอกาสที่ task เดียวต้องรับงานหนักเกินไป แล้วค่อย aggregate กลับตาม key เดิม |
| AQE | Adaptive Query Execution ที่ Spark อาจปรับ physical plan ตอน runtime ตาม statistics จริง เช่น ปรับ join strategy หรือช่วยจัดการ data skew ได้ในบางกรณี |


## Concept explanation

### Shuffle คืออะไร?

ใน Spark ข้อมูลถูกแบ่งเป็นหลาย partition เพื่อให้ executors ทำงาน parallel ได้ แต่บาง operation ต้องการให้ records ที่มี key เดียวกันมาอยู่ด้วยกัน เช่น:

- aggregate ตาม `customer_id`
- join ด้วย `customer_id`
- sort ข้อมูลทั้ง dataset
- deduplicate records

เมื่อ Spark ต้องย้ายข้อมูลข้าม partition เพื่อจัดกลุ่มใหม่ เราเรียกว่า **shuffle**

พูดง่าย ๆ:

> Narrow transformation = ทำงานต่อจาก partition เดิมได้ โดยไม่ต้อง shuffle ข้อมูลข้าม executors  
> Wide transformation = ต้อง redistribute data ข้าม partitions และมักทำให้เกิด shuffle

### ทำไม shuffle แพง?

Shuffle แพงเพราะ Spark ต้อง:

- serialize / deserialize data
- เขียน temporary shuffle files
- ส่งข้อมูลผ่าน network ระหว่าง executors
- sort หรือ hash ข้อมูลตาม key
- รอ task ที่ช้าที่สุดใน stage นั้นให้เสร็จก่อน ซึ่งจะยิ่งช้าถ้ามี data skew
    - Stage คือกลุ่มของ tasks ที่ Spark ต้องรันให้จบก่อนข้ามไป step ถัดไป ถ้ามี task หนึ่งช้ามาก stage ทั้ง stage ก็ต้องรอ task นั้น

ดังนั้น code สั้น ๆ อย่างนี้อาจแพงกว่าที่คิด:

```python
df.groupBy("customer_id").agg(F.sum("amount"))
```

เพราะ Spark ต้องดึง records ของ `customer_id` เดียวกันจาก partition อื่น มาอยู่ใน partition เดียวกันก่อน aggregate

### Broadcast join คืออะไร?

ถ้า join ระหว่าง fact table ใหญ่กับ dimension table เล็ก Spark อาจเลือก **broadcast join** เองได้ หรือเราสามารถใช้ **broadcast hint** เพื่อ suggest ให้ Spark broadcast ฝั่ง dimension ได้

แนวคิดคือ:

1. ส่ง dimension table เล็กไปทุก executor
2. ให้แต่ละ executor join กับ fact partition ของตัวเอง
3. ลด shuffle ฝั่ง fact table ขนาดใหญ่

ตัวอย่าง:

```python
df_fact.join(F.broadcast(df_dim), "customer_id", "left")
```

แต่ broadcast join ไม่ใช่ยาวิเศษ ต้องระวังว่า table ที่ broadcast ต้องเล็กพอจริง ๆ ไม่อย่างนั้นอาจทำให้ executor memory เต็มได้

### Data skew คืออะไร?

Data skew เกิดเมื่อข้อมูลกระจุกที่ key บางค่าเยอะผิดปกติ เช่น customer `101` มี transactions 80% ของทั้ง dataset

ผลคือ แม้ Spark จะมีหลาย partitions แต่ partition ที่รับ records ของ skewed key จะหนักมาก ทำให้บาง task กลายเป็น slow task และลากเวลา execution ของ stage นั้นให้ยาวขึ้น

> Data Skew problem ไม่ใช่แค่จำนวนข้อมูลเยอะ แต่คือข้อมูลกระจายไม่เท่ากัน

### Practical performance mindset

เวลาเจอ join หรือ aggregation ช้า ให้ไล่คิดแบบนี้:

1. Operation นี้เป็น narrow หรือ wide transformation?
   - ถ้าเป็น wide เช่น `join`, `groupBy`, `orderBy`, `repartition` ให้สงสัยเรื่อง shuffle ก่อน

2. `.explain()` เห็น operator อะไร?
   - `Exchange` = มี shuffle
   - `SortMergeJoin` = join strategy ที่ Spark เลือกใช้เมื่อทั้งสอง table ใหญ่เกินจะ broadcast โดยต้อง shuffle และ sort ทั้งสองฝั่งก่อน join จึงมี cost สูง
   - `BroadcastHashJoin` = Spark ใช้ broadcast join แล้ว ให้เช็กว่าฝั่งที่ถูก broadcast เล็กจริงไหม

3. ถ้าเห็น `SortMergeJoin` มี Join ฝั่งไหนเล็กพอจะ force broadcast ได้ไหม?

4. Key distribution กระจุกอยู่ที่ key เดียวหรือไม่?
   - ใช้ `groupBy(key).count()` เพื่อตรวจ skewed key / data skew

5. ถ้ามี data skew ควรแก้ตามสาเหตุ
   - ใช้ broadcast join เมื่อปัญหาอยู่ที่ small dimension join
   - ใช้ salting เมื่อ skewed key ทำให้ aggregation หนักผิดปกติในบาง task
   - ใช้ split processing เมื่อมี skewed key ชัดเจนมาก และต้องการแยก key นั้นออกมา process ต่างหาก


## Mermaid diagram: Shuffle Join, Broadcast Join, and Skew

```mermaid
flowchart TB
    subgraph A[Normal join]
        F1[Large transactions] --> S1[Shuffle by customer_id]
        D1[Customer dimension] --> S2[Shuffle by customer_id]
        S1 --> J1[Join matching partitions on each executor]
        S2 --> J1
    end

    subgraph B[Broadcast join]
        D2[Small customer dimension] --> BC[Broadcast to executors]
        F2[Large transaction partitions] --> J2[Local join on each executor]
        BC --> J2
    end

    subgraph C[Data Skew problem]
        H[Skewed key: customer_id = 101] --> P1[One Spark partition gets too many rows]
        P1 --> T1[One slow task delays the whole stage]
    end
```

Key idea:

> Data skew → skewed key ทำให้ Spark partition บางตัวทำงานหนักกว่า partition อื่นมาก → task นั้นช้ากว่า task อื่น → ทั้ง stage ต้องรอ   

> Broadcast join → table เล็กถูกส่งไปทุก executor ครั้งเดียว → table ใหญ่ไม่ต้อง shuffle เลย → แต่ละ executor ทำ local join ได้ทันที   

> Normal join → ทั้งสอง table ต้อง shuffle ข้อมูลข้าม executor ก่อน แล้วค่อย join partition ที่ตรงกัน → เกิด network transfer ทั้งสองฝั่ง


## Setup / imports


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 3, Finished, Available, Finished, False)

## Create mock data

เราจะสร้าง:

- `df_transactions` เป็น fact-like DataFrame ที่ตั้งใจทำให้เกิด data skew ที่ `customer_id = 101`
- `df_customers` เป็น small dimension DataFrame สำหรับทดลอง normal join และ broadcast join


In [2]:
# Customer 101 is intentionally a skewed key to simulate data skew.
skewed_customer_transactions = [
    (
        f"T-SKEWED-{i:03d}",
        101,
        "P01",
        "Bangkok",
        float(100 + (i % 50)),
        "success",
        "core_banking",
        "2026-05-01",
    )
    for i in range(1, 121)
]

normal_customer_transactions = [
    ("T-201-001", 201, "P02", "Chiang Mai", 850.00, "success", "mobile_app", "2026-05-01"),
    ("T-201-002", 201, "P03", "Chiang Mai", 500.00, "success", "mobile_app", "2026-05-02"),
    ("T-202-001", 202, "P02", "Phuket", 1200.00, "success", "branch", "2026-05-02"),
    ("T-203-001", 203, "P04", "Bangkok", 300.00, "failed", "mobile_app", "2026-05-03"),
    ("T-204-001", 204, "P01", "Rayong", 450.00, "success", "core_banking", "2026-05-03"),
    ("T-205-001", 205, "P03", "Bangkok", 700.00, "pending", "web_portal", "2026-05-04"),
    ("T-206-001", 206, "P04", "Khon Kaen", 640.00, "success", "branch", "2026-05-04"),
    ("T-207-001", 207, "P02", "Bangkok", 980.00, "success", "mobile_app", "2026-05-05"),
]

transactions_data = skewed_customer_transactions + normal_customer_transactions  # Combine two Python lists.

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("region", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("transaction_date", T.StringType(), True),
])

df_transactions = spark.createDataFrame(transactions_data, transaction_schema)

df_transactions.show(10, truncate=False)
df_transactions.printSchema()
print("Transaction count:", df_transactions.count())

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 4, Finished, Available, Finished, False)

+--------------+-----------+----------+-------+------+-------+-------------+----------------+
|transaction_id|customer_id|product_id|region |amount|status |source_system|transaction_date|
+--------------+-----------+----------+-------+------+-------+-------------+----------------+
|T-SKEWED-001  |101        |P01       |Bangkok|101.0 |success|core_banking |2026-05-01      |
|T-SKEWED-002  |101        |P01       |Bangkok|102.0 |success|core_banking |2026-05-01      |
|T-SKEWED-003  |101        |P01       |Bangkok|103.0 |success|core_banking |2026-05-01      |
|T-SKEWED-004  |101        |P01       |Bangkok|104.0 |success|core_banking |2026-05-01      |
|T-SKEWED-005  |101        |P01       |Bangkok|105.0 |success|core_banking |2026-05-01      |
|T-SKEWED-006  |101        |P01       |Bangkok|106.0 |success|core_banking |2026-05-01      |
|T-SKEWED-007  |101        |P01       |Bangkok|107.0 |success|core_banking |2026-05-01      |
|T-SKEWED-008  |101        |P01       |Bangkok|108.0 |succes

In [3]:
customers_data = [
    (101, "Alice", "Bangkok", "premium", "active"),
    (201, "Ben", "Chiang Mai", "standard", "active"),
    (202, "Cara", "Phuket", "standard", "active"),
    (203, "Dan", "Bangkok", "standard", "inactive"),
    (204, "Eve", "Rayong", "standard", "active"),
    (205, "Fah", "Bangkok", "standard", "active"),
    (206, "Gun", "Khon Kaen", "standard", "active"),
    (207, "Hana", "Bangkok", "standard", "active"),
]

customer_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("home_region", T.StringType(), True),
    T.StructField("customer_tier", T.StringType(), True),
    T.StructField("customer_status", T.StringType(), True),
])

df_customers = spark.createDataFrame(customers_data, customer_schema)

df_customers.show(truncate=False)
df_customers.printSchema()
print("Customer dimension count:", df_customers.count())

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 5, Finished, Available, Finished, False)

+-----------+-------------+-----------+-------------+---------------+
|customer_id|customer_name|home_region|customer_tier|customer_status|
+-----------+-------------+-----------+-------------+---------------+
|101        |Alice        |Bangkok    |premium      |active         |
|201        |Ben          |Chiang Mai |standard     |active         |
|202        |Cara         |Phuket     |standard     |active         |
|203        |Dan          |Bangkok    |standard     |inactive       |
|204        |Eve          |Rayong     |standard     |active         |
|205        |Fah          |Bangkok    |standard     |active         |
|206        |Gun          |Khon Kaen  |standard     |active         |
|207        |Hana         |Bangkok    |standard     |active         |
+-----------+-------------+-----------+-------------+---------------+

root
 |-- customer_id: integer (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- home_region: string (nullable = true)
 |-- customer_tier: s

## PySpark code examples

ใน section นี้จะไล่จากการเห็น data skew ในข้อมูล → เห็น shuffle จาก aggregation → เปรียบเทียบ normal join กับ broadcast join → ทดลอง salting เพื่อกระจาย skewed key ใน aggregation

### Example 1: Check key distribution before tuning

ก่อน tune join หรือ aggregation ควรดู distribution ของ key ที่ใช้ก่อน เพราะ data skew มักซ่อนอยู่ใน key เช่น `customer_id`

Cell นี้นับจำนวน transaction ต่อ customer และคำนวณ percentage ของแต่ละ key เทียบกับจำนวน records ทั้งหมด


In [4]:
total_transaction_count = df_transactions.count()

df_customer_distribution = (
    df_transactions
    .groupBy("customer_id")
    .agg(F.count("*").alias("transaction_count"))
    .withColumn(
        "row_share_pct",
        F.round(F.col("transaction_count") * F.lit(100.0) / F.lit(total_transaction_count), 2)
    )
    .orderBy(F.desc("transaction_count"))
)

df_customer_distribution.show(truncate=False)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 6, Finished, Available, Finished, False)

+-----------+-----------------+-------------+
|customer_id|transaction_count|row_share_pct|
+-----------+-----------------+-------------+
|101        |120              |93.75        |
|201        |2                |1.56         |
|206        |1                |0.78         |
|205        |1                |0.78         |
|207        |1                |0.78         |
|202        |1                |0.78         |
|203        |1                |0.78         |
|204        |1                |0.78         |
+-----------+-----------------+-------------+



### Example 2: Aggregation is a wide transformation

`groupBy("customer_id")` ต้องพา records ที่มี `customer_id` เดียวกันไปอยู่ด้วยกัน จึงมักเกิด shuffle

ลองดู `.explain(mode="formatted")` แล้วหา keyword เช่น `Exchange` หรือ aggregate step ใน physical plan


In [5]:
df_customer_revenue = (
    df_transactions
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
    )
    .orderBy(F.desc("transaction_count"))
)

df_customer_revenue.explain(mode="formatted")
df_customer_revenue.show(truncate=False)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 7, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [8]: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733]
Arguments: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [2]: [customer_id#727, amount#730]
Input [8]: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733]

(3) HashAggregate
Input [2]: [customer_id#727, amount#730]
Keys [1]: [customer_id#727]
Functions [3]: [partial_count(1), partial_sum(amount#730)

### Example 3: Inspect partition distribution after repartition by join key

`F.spark_partition_id()` ใช้ตรวจสอบ partition id ของแต่ละ record และเมื่อนำไป `groupBy` จะช่วยดูว่า records กระจายตัวหลัง `repartition` อย่างไร

ในตัวอย่างนี้ เรา repartition ด้วย `customer_id` เพื่อจำลองว่า key เดียวกันถูกส่งไป partition เดียวกัน ถ้า `customer_id = 101` มี records เยอะมาก จะทำให้ partition ที่รับ key นี้มี workload หนักกว่า partition อื่น

> หมายเหตุ:
> - ใช้ `spark_partition_id()` เพื่อ debug เท่านั้น ไม่ควรเก็บเป็น business column ถาวร
> - แม้จะสั่ง `repartition(8, "customer_id")` แต่ผลลัพธ์อาจไม่แสดงครบทั้ง 8 partitions เพราะ Spark ใช้ hash partitioning จาก `customer_id` เพื่อกระจาย records ไปยัง 8 target partitions ซึ่งไม่รับประกันว่าทุก partition จะมี records บาง partition อาจว่าง และหลาย keys อาจ hash ไปตก partition เดียวกันได้

In [6]:
df_partition_distribution_by_customer = (
    df_transactions
    .repartition(8, "customer_id")
    .withColumn("spark_partition_id", F.spark_partition_id())
    .groupBy("spark_partition_id")
    .agg(F.count("*").alias("rows_in_partition"))
    .orderBy("spark_partition_id")
)

df_partition_distribution_by_customer.show(truncate=False)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 8, Finished, Available, Finished, False)

+------------------+-----------------+
|spark_partition_id|rows_in_partition|
+------------------+-----------------+
|0                 |2                |
|2                 |1                |
|3                 |3                |
|4                 |120              |
|6                 |2                |
+------------------+-----------------+



### Example 4: Normal join can require shuffle

เพื่อให้เห็น normal shuffle join ชัดขึ้น cell นี้จะปิด automatic broadcast ชั่วคราวเฉพาะ lab cell นี้ แล้ว join `df_transactions` กับ `df_customers`

ให้ดูใน physical plan ว่ามี `Exchange` และ `SortMergeJoin` หรือไม่

- `Exchange` แปลว่า Spark มี shuffle เกิดขึ้น เพื่อ redistribute data ตาม `customer_id`
- `SortMergeJoin` แปลว่า Spark ใช้ join strategy ที่ต้อง sort ข้อมูลตาม join key ก่อน join
- ถ้าเห็นทั้งสองอย่างพร้อมกัน ให้เข้าใจว่า Spark ต้อง shuffle + sort ข้อมูลก่อน จึงค่อย join partitions ที่มี matching join key บน executor

In [8]:
original_auto_broadcast_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

try:
    # Lab-only setting: disable automatic broadcast so the normal join plan is easier to observe.
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

    df_join_shuffle = (
        df_transactions
        .join(df_customers, on="customer_id", how="left")
    )

    print("Normal join plan with automatic broadcast disabled:")
    df_join_shuffle.explain(mode="formatted")
    print("Joined row count:", df_join_shuffle.count())

finally:
    # Restore the original Spark setting after the lab example.
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", original_auto_broadcast_threshold)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 10, Finished, Available, Finished, False)

Normal join plan with automatic broadcast disabled:
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- SortMergeJoin LeftOuter (7)
      :- Sort (3)
      :  +- Exchange (2)
      :     +- Scan ExistingRDD (1)
      +- Sort (6)
         +- Exchange (5)
            +- Scan ExistingRDD (4)


(1) Scan ExistingRDD
Output [8]: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733]
Arguments: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Exchange
Input [8]: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733]
Arguments: hashpartitioning(customer_id#727, 200), ENSURE_REQUIREMENTS, [plan_id=870]

(3) Sort
Input [8]: [transac

### Example 5: Use broadcast join for a small dimension

ถ้า dimension table เล็กพอ สามารถใช้ `F.broadcast(df_customers)` เพื่อบอก Spark ให้ส่ง dimension ไปทุก executor แล้ว join กับ transaction partitions แบบ local ได้ทันที

สิ่งที่ควรมองหาใน plan:

- `BroadcastExchange` แปลว่า Spark กำลัง broadcast table ฝั่งเล็กไปยัง executors
- `BroadcastHashJoin` แปลว่า Spark ใช้ broadcast join strategy แล้ว
- ถ้าเห็นสองตัวนี้ ให้เข้าใจว่า join นี้ใช้ table เล็กเป็น broadcast side แล้ว join กับ partitions ของ table ใหญ่แบบ local บน executor

Broadcast join เหมาะกับกรณี table ฝั่ง dimension เล็กพอจริง ๆ และไม่ทำให้ executor memory มีปัญหา


In [9]:
df_join_broadcast = (
    df_transactions
    .join(F.broadcast(df_customers), on="customer_id", how="left")
)

print("Broadcast join plan:")
df_join_broadcast.explain(mode="formatted")

df_join_broadcast.select(
    "transaction_id",
    "customer_id",
    "customer_name",
    "customer_tier",
    "amount",
    "status",
).show(10, truncate=False)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 11, Finished, Available, Finished, False)

Broadcast join plan:
== Physical Plan ==
AdaptiveSparkPlan (6)
+- Project (5)
   +- BroadcastHashJoin LeftOuter BuildRight (4)
      :- Scan ExistingRDD (1)
      +- BroadcastExchange (3)
         +- Scan ExistingRDD (2)


(1) Scan ExistingRDD
Output [8]: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733]
Arguments: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Scan ExistingRDD
Output [5]: [customer_id#789, customer_name#790, home_region#791, customer_tier#792, customer_status#793]
Arguments: [customer_id#789, customer_name#790, home_region#791, customer_tier#792, customer_status#793], MapPartitionsRDD[56] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0

### Example 6: Apply simple salting to reduce data skew in aggregation

Salting เป็น pattern หนึ่งที่ใช้เพิ่ม salt column ชั่วคราว เพื่อกระจาย records ของ skewed key ออกเป็นหลาย salt values แล้วทำ partial aggregation ก่อนค่อย aggregate กลับเป็น final result

แนวคิด:

1. เพิ่ม `salt_bucket` เป็น salt column ชั่วคราวจาก hash ของ `transaction_id`
2. aggregate รอบแรกด้วย `customer_id + salt_bucket` เพื่อกระจาย workload ของ skewed key
3. aggregate รอบสุดท้ายกลับมาตาม `customer_id` เพื่อให้ได้ final result ตาม key เดิม
4. validate ว่าผลรวมตรงกับ unsalted aggregation

ตัวอย่างนี้เป็น mock data ขนาดเล็ก จึงไม่ได้ทำให้ performance ดีขึ้นจริงใน notebook แต่ช่วยให้เข้าใจว่า salting ใช้กระจาย workload ของ skewed key เพื่อลดผลกระทบจาก data skew ใน aggregation ได้อย่างไร

In [10]:
salt_bucket_count = 8

df_transactions_salted = (
    df_transactions
    .withColumn(
        "salt_bucket",
        F.pmod(F.xxhash64(F.col("transaction_id")), F.lit(salt_bucket_count)).cast("int")
    )
)

# Tips:
# - xxhash64(column) produces a stable, deterministic hash value. The same input always gives the same hash value,
#   unlike F.rand() which produces a different value every time Spark computes the column.
# - pmod(dividend, divisor) maps the dividend into the range 0 to divisor - 1.
#   pmod (positive modulo) is preferred over % (modulo) because it always returns a non-negative remainder,
#   and the remainder of division is always less than the divisor.

print("Salt bucket distribution for the skewed customer:")
(
    df_transactions_salted
    .filter(F.col("customer_id") == 101)
    .groupBy("salt_bucket")
    .agg(F.count("*").alias("rows_in_bucket"))
    .orderBy("salt_bucket")
    .show(truncate=False)
)

print("Partition distribution after repartition by customer_id + salt_bucket:")
(
    df_transactions_salted
    # Repartition by salted key so the skewed customer can be spread across multiple partitions.
    # Compare this with Example 3, where repartitioning only by customer_id kept the skewed key together.
    .repartition(8, "customer_id", "salt_bucket")
    .withColumn("spark_partition_id", F.spark_partition_id())
    .groupBy("spark_partition_id")
    .agg(F.count("*").alias("rows_in_partition"))
    .orderBy("spark_partition_id")
    .show(truncate=False)
)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 12, Finished, Available, Finished, False)

Salt bucket distribution for the skewed customer:
+-----------+--------------+
|salt_bucket|rows_in_bucket|
+-----------+--------------+
|0          |17            |
|1          |11            |
|2          |17            |
|3          |16            |
|4          |6             |
|5          |13            |
|6          |21            |
|7          |19            |
+-----------+--------------+

Partition distribution after repartition by customer_id + salt_bucket:
+------------------+-----------------+
|spark_partition_id|rows_in_partition|
+------------------+-----------------+
|0                 |2                |
|2                 |17               |
|3                 |2                |
|4                 |45               |
|5                 |24               |
|6                 |2                |
|7                 |36               |
+------------------+-----------------+



In [11]:
df_partial_customer_revenue = (
    df_transactions_salted
    .groupBy("customer_id", "salt_bucket")
    .agg(
        F.count("*").alias("partial_transaction_count"),
        F.sum("amount").alias("partial_total_amount"),
    )
)

df_customer_revenue_salted = (
    df_partial_customer_revenue
    .groupBy("customer_id")
    .agg(
        F.sum("partial_transaction_count").alias("transaction_count"),
        F.round(F.sum("partial_total_amount"), 2).alias("total_amount"),
    )
    .orderBy(F.desc("transaction_count"))
)

df_revenue_compare = (
    df_customer_revenue
    .select(
        "customer_id",
        F.col("transaction_count").alias("unsalted_transaction_count"),
        F.col("total_amount").alias("unsalted_total_amount"),
    )
    .join(
        df_customer_revenue_salted.select(
            "customer_id",
            F.col("transaction_count").alias("salted_transaction_count"),
            F.col("total_amount").alias("salted_total_amount"),
        ),
        on="customer_id",
        how="inner",
    )
    .withColumn(
        "is_count_matched",
        F.col("unsalted_transaction_count") == F.col("salted_transaction_count")
    )
    .withColumn(
        "is_amount_matched",
        F.col("unsalted_total_amount") == F.col("salted_total_amount")
    )
    .orderBy(F.desc("unsalted_transaction_count"))
)

df_revenue_compare.show(truncate=False)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 13, Finished, Available, Finished, False)

+-----------+--------------------------+---------------------+------------------------+-------------------+----------------+-----------------+
|customer_id|unsalted_transaction_count|unsalted_total_amount|salted_transaction_count|salted_total_amount|is_count_matched|is_amount_matched|
+-----------+--------------------------+---------------------+------------------------+-------------------+----------------+-----------------+
|101        |120                       |14660.0              |120                     |14660.0            |true            |true             |
|201        |2                         |1350.0               |2                       |1350.0             |true            |true             |
|206        |1                         |640.0                |1                       |640.0              |true            |true             |
|205        |1                         |700.0                |1                       |700.0              |true            |true             |

### Example 7: Write final summary to a Lakehouse table

Cell นี้เขียน final salted revenue summary เป็น table ชื่อ `day26_customer_revenue_summary`


In [12]:
table_name = "day26_customer_revenue_summary"

(
    df_customer_revenue_salted
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"Table written successfully: {table_name}")
spark.read.table(table_name).show(truncate=False)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 14, Finished, Available, Finished, False)

Table written successfully: day26_customer_revenue_summary
+-----------+-----------------+------------+
|customer_id|transaction_count|total_amount|
+-----------+-----------------+------------+
|101        |120              |14660.0     |
|201        |2                |1350.0      |
|206        |1                |640.0       |
|205        |1                |700.0       |
|207        |1                |980.0       |
|202        |1                |1200.0      |
|203        |1                |300.0       |
|204        |1                |450.0       |
+-----------+-----------------+------------+



## Quick recap

| Question | Answer |
|---|---|
| Shuffle คืออะไร? | การ redistribute data ข้าม partitions / executors เพื่อจัดข้อมูลตาม key ใหม่ |
| Operation ไหนมักทำให้เกิด shuffle? | `groupBy`, `join`, `distinct`, `orderBy`, `repartition` |
| Broadcast join ช่วยอะไร? | ลด shuffle ฝั่ง table ใหญ่โดยส่ง table เล็กไปทุก executor |
| Broadcast join ใช้ได้ตลอดไหม? | ไม่ ควรใช้เมื่อฝั่ง broadcast เล็กพอจริง ๆ และไม่เสี่ยง memory pressure บน executor |
| Data skew คืออะไร? | ข้อมูลกระจุกที่ key หรือ partition บางอันมากผิดปกติ ทำให้บาง task ช้ามาก |
| Data skew ดูจากอะไร? | key distribution, partition distribution, และ runtime task metrics ใน Spark UI |


## Exercises


### Exercise 1: Detect skewed customer keys

สร้าง DataFrame ชื่อ `df_skewed_customer_check` จาก `df_transactions`

Requirements:

- group by `customer_id`
- count transactions ต่อ customer
- คำนวณ `row_share_pct`
- เพิ่ม column `is_skewed_key` โดยให้เป็น `true` เมื่อ `row_share_pct >= 50`
- order by `transaction_count` จากมากไปน้อย

Expected result:

- `customer_id = 101` ควรถูก flag เป็น skewed key
- เห็นชัดว่า row share ของ key นี้สูงกว่าคนอื่นมาก


In [13]:
exercise_total_count = df_transactions.count()

df_skewed_customer_check = (
    df_transactions
    .groupBy("customer_id")
    .agg(F.count("*").alias("transaction_count"))
    .withColumn(
        "row_share_pct",
        F.round(F.col("transaction_count") * F.lit(100.0) / F.lit(exercise_total_count), 2)
    )
    .withColumn("is_skewed_key", F.col("row_share_pct") >= F.lit(50.0))
    .orderBy(F.desc("transaction_count"))
)

df_skewed_customer_check.show(truncate=False)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 15, Finished, Available, Finished, False)

+-----------+-----------------+-------------+-------------+
|customer_id|transaction_count|row_share_pct|is_skewed_key|
+-----------+-----------------+-------------+-------------+
|101        |120              |93.75        |true         |
|201        |2                |1.56         |false        |
|206        |1                |0.78         |false        |
|205        |1                |0.78         |false        |
|207        |1                |0.78         |false        |
|202        |1                |0.78         |false        |
|203        |1                |0.78         |false        |
|204        |1                |0.78         |false        |
+-----------+-----------------+-------------+-------------+



### Exercise 2: Compare normal join and broadcast join

ใช้ `df_transactions` join กับ active customers จาก `df_customers`

Requirements:

- สร้าง `df_active_customers` โดย filter `customer_status = "active"`
- สร้าง `df_exercise_normal_join` โดย join แบบปกติ
- สร้าง `df_exercise_broadcast_join` โดยใช้ `F.broadcast(df_active_customers)`
- ใช้ `.explain(mode="formatted")` ทั้งสองแบบ
- ใช้ `.count()` เพื่อ trigger execution และ validate row count

Expected result:

- broadcast join plan ควรเห็นทั้ง `BroadcastExchange` และ `BroadcastHashJoin`
- row count ของ join ทั้งสองแบบควรเท่ากัน


In [14]:
df_active_customers = df_customers.filter(F.col("customer_status") == "active")

df_exercise_normal_join = (
    df_transactions
    .join(df_active_customers, on="customer_id", how="left")
)

df_exercise_broadcast_join = (
    df_transactions
    .join(F.broadcast(df_active_customers), on="customer_id", how="left")
)

print("Normal join plan:")
df_exercise_normal_join.explain(mode="formatted")

print("Broadcast join plan:")
df_exercise_broadcast_join.explain(mode="formatted")

normal_join_count = df_exercise_normal_join.count()
broadcast_join_count = df_exercise_broadcast_join.count()

print("Normal join count:", normal_join_count)
print("Broadcast join count:", broadcast_join_count)
print("Counts matched:", normal_join_count == broadcast_join_count)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 16, Finished, Available, Finished, False)

Normal join plan:
== Physical Plan ==
AdaptiveSparkPlan (10)
+- Project (9)
   +- SortMergeJoin LeftOuter (8)
      :- Sort (3)
      :  +- Exchange (2)
      :     +- Scan ExistingRDD (1)
      +- Sort (7)
         +- Exchange (6)
            +- Filter (5)
               +- Scan ExistingRDD (4)


(1) Scan ExistingRDD
Output [8]: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733]
Arguments: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Exchange
Input [8]: [transaction_id#726, customer_id#727, product_id#728, region#729, amount#730, status#731, source_system#732, transaction_date#733]
Arguments: hashpartitioning(customer_id#727, 200), ENSURE_REQUIREMENTS, [plan_id=2296]

(3) Sort
Input [8]: [transactio

### Exercise 3: Use salting and validate final result

สร้าง salted aggregation ใหม่โดยใช้ `salt_bucket_count = 4`

Requirements:

- เพิ่ม column `salt_bucket` จาก hash ของ `transaction_id`
- aggregate รอบแรกด้วย `customer_id`, `salt_bucket`
- aggregate รอบสุดท้ายกลับเป็น `customer_id`
- compare กับ unsalted aggregation เดิม
- validate ว่า count และ amount ตรงกัน

Expected result:

- final salted aggregation ควรได้จำนวน transactions และ total amount เท่ากับ unsalted result
- เข้าใจว่า salting ต้องมีขั้นตอน recombine กลับ ไม่ใช่แค่เพิ่ม salt แล้วจบ


In [15]:
exercise_salt_bucket_count = 4

df_exercise_salted = (
    df_transactions
    .withColumn(
        "salt_bucket",
        F.pmod(F.xxhash64(F.col("transaction_id")), F.lit(exercise_salt_bucket_count)).cast("int")
    )
)

df_exercise_partial = (
    df_exercise_salted
    .groupBy("customer_id", "salt_bucket")
    .agg(
        F.count("*").alias("partial_transaction_count"),
        F.sum("amount").alias("partial_total_amount"),
    )
)

df_exercise_salted_final = (
    df_exercise_partial
    .groupBy("customer_id")
    .agg(
        F.sum("partial_transaction_count").alias("salted_transaction_count"),
        F.round(F.sum("partial_total_amount"), 2).alias("salted_total_amount"),
    )
)

df_exercise_unsalted = (
    df_transactions
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("unsalted_transaction_count"),
        F.round(F.sum("amount"), 2).alias("unsalted_total_amount"),
    )
)

df_exercise_validation = (
    df_exercise_unsalted
    .join(df_exercise_salted_final, on="customer_id", how="inner")
    .withColumn(
        "is_count_matched",
        F.col("unsalted_transaction_count") == F.col("salted_transaction_count")
    )
    .withColumn(
        "is_amount_matched",
        F.col("unsalted_total_amount") == F.col("salted_total_amount")
    )
    .orderBy(F.desc("unsalted_transaction_count"))
)

df_exercise_validation.show(truncate=False)

StatementMeta(, 5112cb9e-c0a5-4ce8-9b3b-44fd94859a76, 17, Finished, Available, Finished, False)

+-----------+--------------------------+---------------------+------------------------+-------------------+----------------+-----------------+
|customer_id|unsalted_transaction_count|unsalted_total_amount|salted_transaction_count|salted_total_amount|is_count_matched|is_amount_matched|
+-----------+--------------------------+---------------------+------------------------+-------------------+----------------+-----------------+
|101        |120                       |14660.0              |120                     |14660.0            |true            |true             |
|201        |2                         |1350.0               |2                       |1350.0             |true            |true             |
|206        |1                         |640.0                |1                       |640.0              |true            |true             |
|205        |1                         |700.0                |1                       |700.0              |true            |true             |

## Common mistakes

1. **คิดว่า join ทุกแบบเหมือนกัน**

   Join code อาจหน้าตาคล้ายกัน แต่ physical plan ต่างกันมาก เช่น `SortMergeJoin` กับ `BroadcastHashJoin` ใช้ resource คนละแบบ

2. **Broadcast table ใหญ่เกินไป**

   Broadcast join เหมาะกับ dimension table เล็ก ๆ ถ้า broadcast table ใหญ่เกินไปอาจทำให้ executor เกิด memory pressure หรือ job fail

3. **คิดว่า broadcast variable คือ broadcast join**

   Broadcast variable เป็น Spark mechanism สำหรับส่ง read-only object ไปทุก executor เพื่อใช้ใน UDF หรือ custom logic ส่วน broadcast join คือ join strategy ของ Spark SQL/DataFrame ที่ส่ง table เล็กไปทุก executor เพื่อ join แบบ local

4. **ดูแค่จำนวน rows รวม แต่ไม่ดู distribution**

   Dataset มี 1 ล้าน rows อาจไม่ช้าเท่า dataset 2 แสน rows ที่มี skewed key กระจุกหนักมาก เพราะ data skew ทำให้บาง task ช้าผิดปกติ

5. **ใช้ `spark_partition_id()` เป็น business logic**

   `spark_partition_id()` ใช้เพื่อตรวจสอบการกระจายตัวของ records ตอนรัน job เท่านั้น ไม่ควรใช้เป็น business column เพราะค่า partition id เปลี่ยนได้ตาม physical execution plan, shuffle และจำนวน partitions

6. **ใส่ salt แล้วลืม aggregate กลับ**

   Salting สำหรับ aggregation ต้องมี partial aggregation แล้ว aggregate กลับตาม key เดิม ไม่อย่างนั้นผลลัพธ์จะยังแตกเป็นหลาย rows ตาม `salt_bucket`

7. **แก้ performance ด้วย `repartition()` แบบสุ่ม ๆ**

   `repartition()` เองก็ทำให้เกิด shuffle ถ้าใช้โดยไม่รู้เหตุผลอาจทำให้ job ช้ากว่าเดิม


## Expected Output / Success Criteria

เมื่อจบ Day 26 ควรทำได้:

- อธิบายได้ว่า **shuffle** คือการ redistribute data ข้าม partitions / executors
- แยกได้ว่า operation ไหนเป็น **wide transformation** เช่น `groupBy`, `join`, `distinct`, `orderBy`, `repartition`
- ใช้ `.explain(mode="formatted")` เพื่อมองหา `Exchange`, `BroadcastExchange`, `BroadcastHashJoin`, หรือ join strategy อื่นได้
- เปรียบเทียบ normal join กับ broadcast join ได้ในระดับ practical usage
- อธิบายได้ว่า broadcast join เหมาะกับ small dimension table แต่ไม่ควร broadcast table ใหญ่
- ตรวจ skewed key เบื้องต้นจาก key distribution ได้
- ใช้ `F.spark_partition_id()` ร่วมกับ `groupBy` เพื่อตรวจสอบ partition distribution ของ records ได้
- อธิบาย data skew ได้ว่าเป็นปัญหา distribution ไม่เท่ากัน ไม่ใช่แค่ row count เยอะ
- ทำ simple salting เพื่อลด data skew ใน aggregation และ validate ผลลัพธ์หลัง aggregate กลับได้


## Final takeaway

Shuffle, broadcast join และ data skew เป็น performance mindset ที่ Data Engineer ต้องเริ่มอ่านให้ออก:

> Spark job ที่ช้าไม่ได้แปลว่า code ผิดเสมอไป แต่อาจเกิดจาก data movement, join strategy หรือ data distribution ที่ไม่สมดุล

สำหรับ Day 26 สิ่งที่ต้องจำคือ:

- `groupBy` และ `join` มักต้องคิดเรื่อง shuffle
- broadcast join ใช้กับ dimension table เล็ก ๆ เพื่อช่วยลด shuffle ได้
- data skew คือปัญหาที่ข้อมูลหรือ workload กระจุกไม่สมดุล ทำให้บาง partition หรือบาง task หนักกว่าที่อื่นมาก
- ใช้ `.explain()` เพื่อดู shuffle และ join strategy แล้วดู key distribution / partition distribution คู่กันเพื่อหา data skew
- salting เป็น pattern ที่ช่วยกระจาย workload ของ skewed key ใน aggregation แต่ต้อง aggregate กลับตาม key เดิมและ validate result ทุกครั้ง


## Optional cleanup


In [ ]:
# spark.sql("DROP TABLE IF EXISTS day26_customer_revenue_summary")